In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-72a5708f-f726-47a2-a60d-e3232a1e5eb8;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 418ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
# Load source datasets
orders_reviews = spark.read.csv("s3a://last-mile-optimization-trusted-sofhia/dataset-orders/join_orders_and_reviews.csv/", header=True, inferSchema=True)
customers_geo = spark.read.csv("s3a://last-mile-optimization-trusted-sofhia/dataset-orders/join_customers_and_geolocation.csv/", header=True, inferSchema=True)
sellers_geo = spark.read.csv("s3a://last-mile-optimization-trusted-sofhia/dataset-orders/join_sellers_and_geolocation.csv/", header=True, inferSchema=True)

# Rename geolocation columns to avoid collisions after join
customers_geo = customers_geo.withColumnRenamed("geolocation_lat", "customer_geolocation_lat").withColumnRenamed("geolocation_lng", "customer_geolocation_lng")
sellers_geo = sellers_geo.withColumnRenamed("geolocation_lat", "seller_geolocation_lat").withColumnRenamed("geolocation_lng", "seller_geolocation_lng")

# Join by customer_id and seller_id
orders_customers = orders_reviews.join(customers_geo, on="customer_id", how="inner")
orders_customers_sellers = orders_customers.join(sellers_geo, on="seller_id", how="inner")


from pyspark.sql import functions as F

print("=== VALIDAÇÃO DOS JOINS ===")

print("Total de linhas:", orders_customers_sellers.count())

print(
    "Pedidos únicos:",
    orders_customers_sellers.select("order_id").distinct().count()
)

print(
    "Itens distintos:",
    orders_customers_sellers.select(
        "order_id", "order_item_id"
    ).distinct().count()
)

orders_customers_sellers.printSchema()
orders_customers_sellers.show(5)

26/11/03 01:18:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


=== VALIDAÇÃO DOS JOINS ===


Total de linhas: 35072


Pedidos únicos: 30525


Itens distintos: 35072
root
 |-- seller_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- approval_date_missing: integer (nullable = true)
 |-- approval_date_inconsistent: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- freight_outlier_flag: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- volume_cm3: double (nullable = true)
 |-- review_score: d

26/11/03 01:19:22 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------------------+--------------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------------+--------------------------+--------------------+-------------+-------------------+------+-------------+--------------------+---------------+----------------+----------+------------+----------------------+------------------------+--------------------+---------------+------------------------+------------------------+----------------------+-----------+----------------------+----------------------+
|           seller_id|         customer_id|            order_id|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|approval_date_missing|approval_date_inconsistent|          product_id|order_item_id|shipping_limit_date| price|freight_value|freight_outlier_flag|  category_name|product_weight_g|volum

In [4]:
from pyspark.sql import functions as F
import csv
import os
from datetime import datetime

# Normalize delivery dates and compute on-time flag (1 = on time, 0 = late)
joined_df = orders_customers_sellers
joined_df = joined_df.withColumn(
    "order_delivered_customer_date",
    F.to_timestamp("order_delivered_customer_date")
 ).withColumn(
    "order_estimated_delivery_date",
    F.to_timestamp("order_estimated_delivery_date")
 ).withColumn(
    "delivered_on_time",
    F.when(
        F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"),
        F.lit(1)
    ).otherwise(F.lit(0))
 )

joined_df.printSchema()
print("rows:", joined_df.count())

root
 |-- seller_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- approval_date_missing: integer (nullable = true)
 |-- approval_date_inconsistent: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- freight_outlier_flag: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- volume_cm3: double (nullable = true)
 |-- review_score: double (nullable = true)

rows: 35072


In [5]:
joined_df.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted-sofhia/dataset-orders/final_table_.csv')

spark.stop()

26/11/03 01:19:29 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/03 01:19:30 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
